In [ ]:
# =====================================================
# ✅ 회귀 모델 비교 전체 코드 (preprocessor 포함)
# =====================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor

# -----------------------------------------------------
# 기본 설정
# -----------------------------------------------------
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

def RMSE(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


# =====================================================
# 0) 데이터 준비 (이미 있으면 이 부분은 생략 가능)
# =====================================================
# X_train, X_test, y_train, y_test
# 는 이미 만들어져 있다고 가정
# (없으면 여기서 만들어야 함)


# =====================================================
# 1) 전처리 정의 (🔥 이게 빠져서 에러 난 것)
# =====================================================

# 🔹 수치형 / 범주형 컬럼 (네 프로젝트 기준 예시)
num_cols = [
    "dep_hour",
    "dep_minute",
    "dep_weekday",
    "is_weekend",
    "기온(°C)",
    "풍속_ms"
]

cat_cols = [
    "항공사",
    "출발지",
    "arrival_code",
    "flight_type"
]

# 실제 존재하는 컬럼만 사용
num_cols = [c for c in num_cols if c in X_train.columns]
cat_cols = [c for c in cat_cols if c in X_train.columns]

# OneHotEncoder 버전 호환
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

preprocessor = ColumnTransformer(
    transformers=[
        # 범주형: 결측 → UNKNOWN → OneHot
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
            ("ohe", ohe)
        ]), cat_cols),

        # 수치형: 결측 → 중앙값
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), num_cols)
    ],
    remainder="drop"
)

print("✅ preprocessor 생성 완료")
print(" - num_cols:", num_cols)
print(" - cat_cols:", cat_cols)


# =====================================================
# 2) 모델 정의
# =====================================================

lin_reg = Pipeline([
    ("prep", preprocessor),
    ("reg", LinearRegression())
])

ridge_reg = Pipeline([
    ("prep", preprocessor),
    ("reg", Ridge(alpha=1.0))
])

lgbm_reg = Pipeline([
    ("prep", preprocessor),
    ("reg", LGBMRegressor(
        objective="regression",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ))
])


# =====================================================
# 3) 학습 + 예측
# =====================================================

# Linear
lin_reg.fit(X_train, y_train)
y_pred_lin = lin_reg.predict(X_test)

# Ridge
ridge_reg.fit(X_train, y_train)
y_pred_ridge = ridge_reg.predict(X_test)

# Log-Ridge
y_train_log = np.log1p(y_train)
ridge_reg.fit(X_train, y_train_log)
y_pred_log_ridge = np.expm1(ridge_reg.predict(X_test))

# LightGBM Raw
lgbm_reg.fit(X_train, y_train)
y_pred_lgbm_raw = lgbm_reg.predict(X_test)

# LightGBM Log
lgbm_reg.fit(X_train, y_train_log)
y_pred_lgbm_log = np.expm1(lgbm_reg.predict(X_test))


# =====================================================
# 4) LightGBM 최종 선택 (MAE 기준)
# =====================================================

if mean_absolute_error(y_test, y_pred_lgbm_log) <= mean_absolute_error(y_test, y_pred_lgbm_raw):
    y_pred_lgbm = y_pred_lgbm_log
    print("✅ LightGBM 선택: Log Target")
else:
    y_pred_lgbm = y_pred_lgbm_raw
    print("✅ LightGBM 선택: Raw Target")


# =====================================================
# 5) 성능 요약표
# =====================================================

summary_df = pd.DataFrame({
    "모델": [
        "Linear Regression",
        "Ridge Regression",
        "Log-Ridge Regression",
        "LightGBM"
    ],
    "MAE (분)": [
        mean_absolute_error(y_test, y_pred_lin),
        mean_absolute_error(y_test, y_pred_ridge),
        mean_absolute_error(y_test, y_pred_log_ridge),
        mean_absolute_error(y_test, y_pred_lgbm)
    ],
    "RMSE (분)": [
        RMSE(y_test, y_pred_lin),
        RMSE(y_test, y_pred_ridge),
        RMSE(y_test, y_pred_log_ridge),
        RMSE(y_test, y_pred_lgbm)
    ]
}).sort_values("MAE (분)").reset_index(drop=True)

display(summary_df)


In [1]:
# =====================================================
# ✅ 모델 학습/평가/비교 (최종 정리 버전)
# - Linear / Ridge / Log-Ridge / LightGBM
# - LightGBM은 Raw vs Log 중 MAE 기준으로 자동 선택
# - summary_df, preds 생성
# - 비교 시각화 3종 포함
# =====================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline

# ✅ 폰트
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


def RMSE(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


# =====================================================
# 1) 모델 정의
# =====================================================

lin_reg = Pipeline([
    ("prep", preprocessor),
    ("reg", LinearRegression())
])

ridge_reg = Pipeline([
    ("prep", preprocessor),
    ("reg", Ridge(alpha=1.0))
])

lgbm_reg = Pipeline([
    ("prep", preprocessor),
    ("reg", LGBMRegressor(
        objective="regression",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ))
])


# =====================================================
# 2) 학습 + 예측
# =====================================================

# (1) Linear
lin_reg.fit(X_train, y_train)
y_pred_lin = lin_reg.predict(X_test)

# (2) Ridge
ridge_reg.fit(X_train, y_train)
y_pred_ridge = ridge_reg.predict(X_test)

# (3) Log-Ridge
y_train_log = np.log1p(y_train)

ridge_reg.fit(X_train, y_train_log)
y_pred_log_ridge_log = ridge_reg.predict(X_test)
y_pred_log_ridge = np.expm1(y_pred_log_ridge_log)

# (4) LightGBM Raw
lgbm_reg.fit(X_train, y_train)
y_pred_lgbm_raw = lgbm_reg.predict(X_test)

# (5) LightGBM Log Target
lgbm_reg.fit(X_train, y_train_log)
y_pred_lgbm_logspace = lgbm_reg.predict(X_test)
y_pred_lgbm_log = np.expm1(y_pred_lgbm_logspace)


# =====================================================
# 3) ✅ LightGBM Raw vs Log 중 더 좋은 것 자동 선택
# =====================================================

mae_lgbm_raw = mean_absolute_error(y_test, y_pred_lgbm_raw)
mae_lgbm_log = mean_absolute_error(y_test, y_pred_lgbm_log)

if mae_lgbm_log <= mae_lgbm_raw:
    y_pred_lgbm_final = y_pred_lgbm_log
    print("✅ LightGBM 최종 선택: Log Target")
else:
    y_pred_lgbm_final = y_pred_lgbm_raw
    print("✅ LightGBM 최종 선택: Raw Target")


# =====================================================
# 4) 성능 요약표 (summary_df)
# =====================================================

summary_df = pd.DataFrame({
    "모델": [
        "Linear Regression",
        "Ridge Regression",
        "Log-Ridge Regression",
        "LightGBM"
    ],
    "MAE (분)": [
        mean_absolute_error(y_test, y_pred_lin),
        mean_absolute_error(y_test, y_pred_ridge),
        mean_absolute_error(y_test, y_pred_log_ridge),
        mean_absolute_error(y_test, y_pred_lgbm_final)
    ],
    "RMSE (분)": [
        RMSE(y_test, y_pred_lin),
        RMSE(y_test, y_pred_ridge),
        RMSE(y_test, y_pred_log_ridge),
        RMSE(y_test, y_pred_lgbm_final)
    ]
}).sort_values("MAE (분)").reset_index(drop=True)

display(summary_df)


# =====================================================
# 5) 모델별 예측값 모음 (preds)
# =====================================================

preds = {
    "Linear Regression": y_pred_lin,
    "Ridge Regression": y_pred_ridge,
    "Log-Ridge Regression": y_pred_log_ridge,
    "LightGBM": y_pred_lgbm_final
}

best_name = summary_df.loc[0, "모델"]
print("✅ best model =", best_name)


# =====================================================
# 6) (시각화 1) 모델 랭킹 막대: MAE
# =====================================================

colors = ["tab:orange" if m == best_name else "lightgray"
          for m in summary_df["모델"]]

plt.figure(figsize=(9,4))
plt.barh(summary_df["모델"], summary_df["MAE (분)"], color=colors)
plt.gca().invert_yaxis()
plt.title(f"모델별 MAE 비교 (1등: {best_name})")
plt.xlabel("MAE(분) (낮을수록 좋음)")
for i, v in enumerate(summary_df["MAE (분)"]):
    plt.text(v, i, f"  {v:.1f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()


# =====================================================
# 7) (시각화 2) 오차 허용치별 적중률 Accuracy@k
# =====================================================

y_true = np.asarray(y_test)
thresholds = np.arange(0, 61, 5)

plt.figure(figsize=(10,4))
for name, p in preds.items():
    abs_err = np.abs(np.asarray(p) - y_true)
    acc = [(abs_err <= t).mean() * 100 for t in thresholds]

    if name == best_name:
        plt.plot(thresholds, acc, marker="o", linewidth=3,
                 label=f"{name} (베스트)")
    else:
        plt.plot(thresholds, acc, alpha=0.35, label=name)

plt.title("오차 허용치(k분)별 적중률 비교")
plt.xlabel("허용 오차 k(분)")
plt.ylabel("적중률(%)  =  |예측-실제| ≤ k")
plt.legend(ncol=2)
plt.tight_layout()
plt.show()


# =====================================================
# 8) (시각화 3) 지연 구간별 MAE
# =====================================================

bins = [0, 10, 30, 60, np.inf]
labels = ["0~10", "10~30", "30~60", "60+"]

plt.figure(figsize=(10,4))
for name, p in preds.items():
    df_err = pd.DataFrame({
        "y_true": y_true,
        "y_pred": p
    })
    df_err["abs_err"] = np.abs(df_err["y_pred"] - df_err["y_true"])
    df_err["delay_bin"] = pd.cut(df_err["y_true"], bins=bins, labels=labels)

    mae_by_bin = df_err.groupby("delay_bin")["abs_err"].mean()

    if name == best_name:
        plt.plot(labels, mae_by_bin, marker="o", linewidth=3,
                 label=f"{name} (베스트)")
    else:
        plt.plot(labels, mae_by_bin, alpha=0.35, label=name)

plt.title("지연 구간별 MAE 비교")
plt.xlabel("실제 지연 구간(분)")
plt.ylabel("MAE(분)")
plt.legend()
plt.tight_layout()
plt.show()


NameError: name 'preprocessor' is not defined